## Anzahl der Publikationen an niedersächsischen Hochschulen

Frage: Wie hoch ist das Publikationsvolumen je Hochschule in Niedersachsen?

In [1]:
from google.cloud import bigquery
import pandas as pd

In [13]:
client = bigquery.Client(project='subugoe-collaborative')

Folgende Szenarien werden evaluiert:

    - Nur OpenAlex (BigQuery)
    
    - OpenAlex mit Bielefelder Institutionskodierung (BigQuery)
    
    - OpenAlex + Bielefelder Institutionskodierung (BigQuery)
    
    - Scopus mit Bielefelder Institutionskodierung (KB)
    
    - OpenAlex mit Bielefelder Institutionskodierung (KB)
    
    - (OpenAlex mit Bielefelder Institutionskodierung) + (Scopus mit Bielefelder Institutionskodierung) (KB)

Abfragen sind zunächst auf die Dokumententypen 'article' und 'review' beschränkt sowie auf die Publikationsjahre 2015 bis 2025.

Bugs in der aktuellen Bielefelder Institutionskodierung:

    - Nicht in der KB-Kodierung enthalten: https://ror.org/02743t710 (Norddeutsche Hochschule für Rechtspflege)
    
    - Hochschule Hannover hat die ROR ID von Fachhochschule für die Wirtschaft Hannover (FHDW)

Zunächst wird eine Liste mit allen Hochschulen in Niedersachsen erstellt (11 Universitäten, 7 Hochschulen und 2 Kunst- und Musikhochschulen). Die Liste wurde mithilfe der Bielefelder Institutionskodierung und GERiT erstellt. Nicht enthalten: Norddeutsche Hochschule für Rechtspflege, da sie nicht in der Institutionskodierung auftaucht. Private Hochschulen werden ausgeschlossen.

In [95]:
liste_aller_hochschulen = client.query(f"""
                                        SELECT DISTINCT inst_name, dfg_inst_id, ror, sector, federal_state
                                        FROM `subugoe-collaborative.resources.inst_with_federal_state` AS f
                                        JOIN `subugoe-collaborative.openbib.kb_inst_lookup` AS kb
                                            ON CASE
                                                WHEN kb.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                                                ELSE kb.ror 
                                            END = CONCAT('https://ror.org/', f.ror_id)
                                        LEFT JOIN UNNEST(current_sectors) AS sector
                                        WHERE federal_state = 'Niedersachsen' 
                                          AND sector IN ('uni', 'fh', 'khmh') 
                                          AND dfg_inst_id NOT IN (
                                            220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                                            13033, --PFH Private Hochschule Göttingen
                                            233118106, -- Leibniz-Fachhochschule
                                            198800578, -- Hochschule 21 Buxtehude
                                            195374963 -- Hochschule Weserbergland
                                          )
                                       """).to_dataframe()

In [96]:
liste_aller_hochschulen

,inst_name,dfg_inst_id,ror,sector,federal_state
0,Leuphana Universität Lüneburg,10232,https://ror.org/02w2y2t16,uni,Niedersachsen
1,Carl von Ossietzky Universität Oldenburg,10233,https://ror.org/033n9gh91,uni,Niedersachsen
2,Jade Hochschule Wilhelmshaven/Oldenburg/Elsfleth,10533,https://ror.org/02vvvm705,fh,Niedersachsen
3,Hochschule Emden/Leer,980710,https://ror.org/01bc76c69,fh,Niedersachsen
4,Gottfried Wilhelm Leibniz Universität Hannover,10238,https://ror.org/0304hq317,uni,Niedersachsen
5,"Hochschule für Musik, Theater und Medien Hannover",10246,https://ror.org/00x67m532,khmh,Niedersachsen
6,Hochschule Hannover,10252,https://ror.org/03z6vda50,fh,Niedersachsen
7,Stiftung Tierärztliche Hochschule Hannover,10249,https://ror.org/015qjqf64,uni,Niedersachsen
8,Medizinische Hochschule Hannover (MHH),10247,https://ror.org/00f2yqf98,uni,Niedersachsen
9,Hochschule für angewandte Wissenschaft und Kun...,10253,https://ror.org/00f5q5839,fh,Niedersachsen


### Nur OpenAlex (Snapshot März 2026)

In [86]:
oal = client.query(f"""
                  SELECT 
                        COUNT(DISTINCT(doi)) AS n,
                        CASE 
                          WHEN inst.ror = 'https://ror.org/021ft0n22' THEN 'Georg-August-Universität Göttingen' -- UMG -> Universität Göttingen
                          ELSE federal_states.inst_name
                        END AS inst_name,
                        oal.publication_year,
                        primary_topic.domain.display_name AS topic_domain,
                        sector
                    FROM `subugoe-collaborative.openalex_walden.works` AS oal, UNNEST(authorships) AS aut, UNNEST(aut.institutions) AS inst
                    JOIN `subugoe-collaborative.resources.inst_with_federal_state` AS federal_states
                        ON inst.ror = CONCAT('https://ror.org/', federal_states.ror_id)
                    JOIN `subugoe-collaborative.openbib.kb_inst_lookup` AS kb
                        ON CASE
                            WHEN kb.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                            ELSE kb.ror 
                        END = CONCAT('https://ror.org/', federal_states.ror_id)
                    LEFT JOIN UNNEST(current_sectors) AS sector
                    WHERE oal.type IN ('article', 'review') 
                        AND is_paratext=FALSE 
                        AND is_retracted=FALSE 
                        AND is_xpac=FALSE
                        AND federal_state = 'Niedersachsen' 
                        AND sector IN ('uni', 'fh', 'khmh') 
                        AND dfg_inst_id NOT IN (
                          220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                          13033, --PFH Private Hochschule Göttingen
                          233118106, -- Leibniz-Fachhochschule
                          198800578, -- Hochschule 21 Buxtehude
                          195374963 -- Hochschule Weserbergland
                        )
                        AND publication_year BETWEEN 2015 AND 2025
                    GROUP BY inst_name, publication_year, topic_domain, sector
                    """).to_dataframe()

In [87]:
oal.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False)

,inst_name,count
1,Georg-August-Universität Göttingen,37501
11,Medizinische Hochschule Hannover (MHH),31371
2,Gottfried Wilhelm Leibniz Universität Hannover,23100
15,Technische Universität Braunschweig,17747
0,Carl von Ossietzky Universität Oldenburg,13187
17,Universität Osnabrück,6858
13,Stiftung Tierärztliche Hochschule Hannover,5807
10,Leuphana Universität Lüneburg,5344
16,Technische Universität Clausthal,4314
14,Stiftung Universität Hildesheim,2330


In [88]:
oal.groupby(['topic_domain'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False)

,topic_domain,count
2,Physical Sciences,62878
0,Health Sciences,38977
1,Life Sciences,28841
3,Social Sciences,22433


Eine zusätzliche Anfrage wird für die Norddeutsche Hochschule für Rechtspflege erstellt: 

In [98]:
oal_hr_nord = client.query(f"""
                  SELECT 
                        COUNT(DISTINCT(doi)) AS n,
                        inst.display_name AS inst_name,
                        oal.publication_year,
                        primary_topic.domain.display_name AS topic_domain
                    FROM `subugoe-collaborative.openalex_walden.works` AS oal, UNNEST(authorships) AS aut, UNNEST(aut.institutions) AS inst
                    WHERE oal.type IN ('article', 'review') 
                        AND is_paratext=FALSE 
                        AND is_retracted=FALSE 
                        AND is_xpac=FALSE
                        AND inst.ror = 'https://ror.org/02743t710'
                        AND publication_year BETWEEN 2015 AND 2025
                    GROUP BY inst_name, publication_year, topic_domain
                    """).to_dataframe()

In [99]:
oal_hr_nord

,n,inst_name,publication_year,topic_domain
0,1,Norddeutsche Hochschule für Rechtspflege,2022,Health Sciences


### OpenAlex (Snapshot März 2026 mit Bielefelder Institutionskodierung vom Juli 2025)

Institutionskodierung vom Juli 2025

!!! Publikationsjahr 2025 nur teilweise erfasst

In [66]:
kb = client.query(f"""
                    SELECT 
                        COUNT(DISTINCT(o.doi)) AS n,
                        federal_states.inst_name,
                        o.publication_year,
                        primary_topic.domain.display_name AS topic_domain,
                        sector
                    FROM `subugoe-collaborative.openbib.kb_a_addr_inst` AS inst
                    JOIN `subugoe-collaborative.openbib.kb_inst_lookup` AS kb_inst
                      ON inst.inst_id_top = kb_inst.inst_id
                    JOIN `subugoe-collaborative.resources.inst_with_federal_state` AS federal_states
                      ON CASE
                            WHEN kb_inst.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                            ELSE kb_inst.ror 
                          END = CONCAT('https://ror.org/', federal_states.ror_id)
                    JOIN `subugoe-collaborative.openalex_walden.works` AS o
                        ON inst.openalex_id = o.id
                    LEFT JOIN UNNEST(current_sectors) AS sector
                    WHERE o.type IN ('article', 'review') 
                        AND is_paratext=FALSE 
                        AND is_retracted=FALSE 
                        AND is_xpac=FALSE
                        AND federal_state = 'Niedersachsen' 
                        AND sector IN ('uni', 'fh', 'khmh') 
                        AND dfg_inst_id NOT IN (
                          220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                          13033, --PFH Private Hochschule Göttingen
                          233118106, -- Leibniz-Fachhochschule
                          198800578, -- Hochschule 21 Buxtehude
                          195374963 -- Hochschule Weserbergland
                        )
                        AND publication_year BETWEEN 2015 AND 2025
                    GROUP BY inst_name, publication_year, topic_domain, sector
                    """).to_dataframe()

In [67]:
kb.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False)

,inst_name,count
1,Georg-August-Universität Göttingen,39161
11,Medizinische Hochschule Hannover (MHH),28693
2,Gottfried Wilhelm Leibniz Universität Hannover,21164
15,Technische Universität Braunschweig,15494
0,Carl von Ossietzky Universität Oldenburg,11319
17,Universität Osnabrück,5753
13,Stiftung Tierärztliche Hochschule Hannover,4274
10,Leuphana Universität Lüneburg,4036
16,Technische Universität Clausthal,3831
14,Stiftung Universität Hildesheim,1593


In [68]:
kb.groupby(['topic_domain'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False)

,topic_domain,count
2,Physical Sciences,54422
0,Health Sciences,38951
1,Life Sciences,26675
3,Social Sciences,19031


### OpenAlex und KB (BigQuery)

Diese Abfrage kombiniert die Institutionszuordnung in OpenAlex und der Bielefelder Institutionskodierung

In [69]:
oal_kb = client.query(f"""
                        SELECT 
                          COUNT(DISTINCT(doi)) AS n,
                          inst_name,
                          publication_year,
                          topic_domain,
                          sector
                        FROM (
                          SELECT DISTINCT
                                doi,
                                inst_name,
                                publication_year,
                                topic_domain,
                                sector
                              FROM (
                                SELECT 
                                    o.id,
                                    o.doi,
                                    federal_states.inst_name,
                                    o.publication_year,
                                    primary_topic.domain.display_name AS topic_domain,
                                    sector
                                    -- 'KB' AS source 
                                FROM `subugoe-collaborative.openbib.kb_a_addr_inst` AS inst
                                JOIN `subugoe-collaborative.openbib.kb_inst_lookup` AS kb_inst
                                  ON inst.inst_id_top = kb_inst.inst_id
                                JOIN `subugoe-collaborative.resources.inst_with_federal_state` AS federal_states
                                  ON CASE
                                        WHEN kb_inst.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                                        ELSE kb_inst.ror 
                                      END = CONCAT('https://ror.org/', federal_states.ror_id)
                                JOIN `subugoe-collaborative.openalex_walden.works` AS o
                                    ON inst.openalex_id = o.id
                                LEFT JOIN UNNEST(current_sectors) AS sector
                                WHERE o.type IN ('article', 'review') 
                                    AND is_paratext=FALSE 
                                    AND is_retracted=FALSE 
                                    AND is_xpac=FALSE
                                    AND federal_state = 'Niedersachsen' 
                                    AND sector IN ('uni', 'fh', 'khmh') 
                                    AND dfg_inst_id NOT IN (
                                      220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                                      13033, --PFH Private Hochschule Göttingen
                                      233118106, -- Leibniz-Fachhochschule
                                      198800578, -- Hochschule 21 Buxtehude
                                      195374963 -- Hochschule Weserbergland
                                    )
                                    AND publication_year BETWEEN 2015 AND 2025 
                              UNION DISTINCT
                                SELECT 
                                    oal.id,
                                    doi,
                                    CASE 
                                      WHEN inst.ror = 'https://ror.org/021ft0n22' THEN 'Georg-August-Universität Göttingen' -- UMG -> Universität Göttingen
                                      ELSE federal_states.inst_name
                                    END AS inst_name,
                                    oal.publication_year,
                                    primary_topic.domain.display_name AS topic_domain,
                                    sector
                                    -- 'OAL' AS source 
                                FROM `subugoe-collaborative.openalex_walden.works` AS oal, UNNEST(authorships) AS aut, UNNEST(aut.institutions) AS inst
                                JOIN `subugoe-collaborative.resources.inst_with_federal_state` AS federal_states
                                    ON inst.ror = CONCAT('https://ror.org/', federal_states.ror_id)
                                JOIN `subugoe-collaborative.openbib.kb_inst_lookup` AS kb
                                    ON federal_states.dfg_inst_id = kb.dfg_instituts_id
                                LEFT JOIN UNNEST(current_sectors) AS sector
                                WHERE oal.type IN ('article', 'review') 
                                    AND is_paratext=FALSE 
                                    AND is_retracted=FALSE 
                                    AND is_xpac=FALSE
                                    AND federal_state = 'Niedersachsen' 
                                    AND sector IN ('uni', 'fh', 'khmh') 
                                    AND dfg_inst_id NOT IN (
                                      220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                                      13033, --PFH Private Hochschule Göttingen
                                      233118106, -- Leibniz-Fachhochschule
                                      198800578, -- Hochschule 21 Buxtehude
                                      195374963 -- Hochschule Weserbergland
                                    )
                                    AND publication_year BETWEEN 2015 AND 2025 
                                )
                        )
                        GROUP BY inst_name, publication_year, topic_domain, sector
                """).to_dataframe()

In [70]:
oal_kb.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False)

,inst_name,count
1,Georg-August-Universität Göttingen,46104
11,Medizinische Hochschule Hannover (MHH),31972
2,Gottfried Wilhelm Leibniz Universität Hannover,24275
15,Technische Universität Braunschweig,18447
0,Carl von Ossietzky Universität Oldenburg,13896
17,Universität Osnabrück,7047
13,Stiftung Tierärztliche Hochschule Hannover,5833
10,Leuphana Universität Lüneburg,5376
16,Technische Universität Clausthal,4368
14,Stiftung Universität Hildesheim,2341


In [71]:
oal_kb.groupby(['topic_domain'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False)

,topic_domain,count
2,Physical Sciences,65370
0,Health Sciences,45272
1,Life Sciences,31294
3,Social Sciences,23453


### Nur Scopus (KB)

In [72]:
import psycopg2 as pg
import os
from sqlalchemy import create_engine

In [4]:
host = os.environ['KB_HOST']
database = os.environ['KB_DATABASE']
user = os.environ['KB_USER']
pw = os.environ['KB_PASSWORD']
port = os.environ['KB_PORT']
engine = create_engine(f'postgresql://{user}:{pw}@{host}:{port}/{database}')

In [73]:
scp = pd.read_sql("""
                  SELECT 
                      COUNT(DISTINCT(scp.doi)) AS n,
                      federal_states.inst_name,
                      scp.pubyear,
                      scp.class_name AS topic
                  FROM scp_b_202510.items AS scp
                  JOIN scp_b_202510.add_institution_kb_a AS kb_a
                      ON scp.item_id = kb_a.item_id
                  JOIN scp_b_202510.add_institution_lookup_kb_suppl AS kb_inst
                      ON kb_a.inst_id_top = kb_inst.inst_id
                  JOIN unignhaupka.inst_with_federal_state_filled AS federal_states
                      ON CASE
                            WHEN kb_inst.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                            ELSE kb_inst.ror 
                          END = CONCAT('https://ror.org/', federal_states.ror_id)
                  WHERE item_type && ARRAY['Review', 'Article']
                      AND federal_state = 'Niedersachsen' 
                      AND ('uni' = ANY(current_sectors) OR 'fh' = ANY(current_sectors) OR 'khmh' = ANY(current_sectors))
                      AND dfg_inst_id NOT IN (
                          220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                          13033, --PFH Private Hochschule Göttingen
                          233118106, -- Leibniz-Fachhochschule
                          198800578, -- Hochschule 21 Buxtehude
                          195374963 -- Hochschule Weserbergland
                      )
                      AND pubyear BETWEEN 2015 AND 2025 
                  GROUP BY inst_name, pubyear, topic
                  """, 
                  con=engine)

In [74]:
scp.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False)

,inst_name,count
1,Georg-August-Universität Göttingen,39703
11,Medizinische Hochschule Hannover (MHH),23379
2,Gottfried Wilhelm Leibniz Universität Hannover,17291
15,Technische Universität Braunschweig,12844
0,Carl von Ossietzky Universität Oldenburg,10717
17,Universität Osnabrück,4924
13,Stiftung Tierärztliche Hochschule Hannover,4902
10,Leuphana Universität Lüneburg,3753
16,Technische Universität Clausthal,3121
14,Stiftung Universität Hildesheim,1142


### Nur OpenAlex BDB (KB)

In [75]:
oal_kb_db = pd.read_sql("""
                      SELECT 
                          COUNT(DISTINCT(oal.doi)) AS n,
                          federal_states.inst_name,
                          oal.pubyear,
                          oal.class_name AS topic
                      FROM oal_b_20250711.items AS oal
                      JOIN oal_b_20250711.add_institution_kb_a AS kb_a
                          ON oal.item_id = kb_a.item_id
                      JOIN oal_b_20250711.add_institution_lookup_kb_suppl AS kb_inst
                          ON kb_a.inst_id_top = kb_inst.inst_id
                      JOIN unignhaupka.inst_with_federal_state_filled AS federal_states
                          ON CASE
                                WHEN kb_inst.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                                ELSE kb_inst.ror 
                              END = CONCAT('https://ror.org/', federal_states.ror_id)
                      WHERE item_type && ARRAY['review', 'article']
                          AND federal_state = 'Niedersachsen' 
                          AND ('uni' = ANY(current_sectors) OR 'fh' = ANY(current_sectors) OR 'khmh' = ANY(current_sectors))
                          AND dfg_inst_id NOT IN (
                              220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                              13033, --PFH Private Hochschule Göttingen
                              233118106, -- Leibniz-Fachhochschule
                              198800578, -- Hochschule 21 Buxtehude
                              195374963 -- Hochschule Weserbergland
                          )
                          AND pubyear BETWEEN 2015 AND 2025 
                      GROUP BY inst_name, pubyear, topic
                      """, 
                      con=engine)

In [76]:
oal_kb_db.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False)

,inst_name,count
1,Georg-August-Universität Göttingen,39927
11,Medizinische Hochschule Hannover (MHH),28773
2,Gottfried Wilhelm Leibniz Universität Hannover,21737
15,Technische Universität Braunschweig,15721
0,Carl von Ossietzky Universität Oldenburg,11511
17,Universität Osnabrück,5887
13,Stiftung Tierärztliche Hochschule Hannover,4281
10,Leuphana Universität Lüneburg,4031
16,Technische Universität Clausthal,3854
14,Stiftung Universität Hildesheim,1576


### OpenAlex BDB über AFF-String (KB)

In [105]:
oal_kb_aff_db = pd.read_sql("""
                            with oal as (
                                select
                                	inst.name as inst_name,
                                    doi
                                from oal_b_20250711.items x
                                inner join oal_b_20250711.items_authors via
                                    on x.item_id = via.item_id
                                inner join oal_b_20250711.authors_affiliations vaa2
                                    on x.item_id = vaa2.item_id and via.author_seq_nr = vaa2.author_seq_nr
                                inner join oal_b_20250711.items_affiliations ia
                                    on x.item_id = ia.item_id and vaa2.address_full = ia.address_full
                                inner join oal_b_20250711.add_institution_kb_a kaais
                                    on x.item_id = kaais.item_id and ia.address_full = kaais.address_full
                                inner join oal_b_20250711.add_institution_lookup_kb inst
                                    on kaais.inst_id_top = inst.inst_id
                                inner join oal_b_20250711.add_institution_lookup_kb_suppl AS kb_inst
                                	on kaais.inst_id_top = kb_inst.inst_id
                                inner join unignhaupka.inst_with_federal_state_filled AS federal_states
                                	on case
                                    	when kb_inst.inst_id = 621 then 'https://ror.org/03m2kj587'
                                    else kb_inst.ror 
                                  end = CONCAT('https://ror.org/', federal_states.ror_id)
                                where
                                    item_type && array['article', 'review']
                                    and federal_state = 'Niedersachsen' 
                                    and ('uni' = any(current_sectors) or 'fh' = any(current_sectors) or 'khmh' = any(current_sectors))
                                    and dfg_inst_id not in (
                                    	220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                                      	13033, --PFH Private Hochschule Göttingen
                                      	233118106, -- Leibniz-Fachhochschule
                                        198800578, -- Hochschule 21 Buxtehude
                                        195374963 -- Hochschule Weserbergland
                                  	)
                                  	and pubyear between 2015 and 2025 
                              )
                              select inst_name, count(distinct(doi)) as n
                              from oal
                              group by inst_name
                              order by n desc
                      """, 
                      con=engine)

In [106]:
oal_kb_aff_db

,inst_name,n
0,Georg-August-Universität Göttingen,39926
1,Medizinische Hochschule Hannover (MHH),28773
2,Gottfried Wilhelm Leibniz Universität Hannover,21736
3,Technische Universität Braunschweig,15721
4,Carl von Ossietzky Universität Oldenburg,11511
5,Universität Osnabrück,5887
6,Stiftung Tierärztliche Hochschule Hannover,4281
7,Leuphana Universität Lüneburg,4031
8,Technische Universität Clausthal,3854
9,Stiftung Universität Hildesheim,1576


### Scopus BDB über AFF-String (KB)

In [77]:
scp_aff = pd.read_sql("""
                  with scopus as (
                    select
                    	inst.name as inst_name,
                        doi
                    from scp_b_202510.items x
                    inner join scp_b_202510.items_authors via
                        on x.item_id = via.item_id
                    inner join scp_b_202510.authors_affiliations vaa2
                        on x.item_id = vaa2.item_id and via.author_seq_nr = vaa2.author_seq_nr
                    inner join scp_b_202510.items_affiliations ia
                        on x.item_id = ia.item_id and vaa2.aff_seq_nr = ia.aff_seq_nr
                    inner join scp_b_202510.add_institution_kb_a kaais
                        on x.item_id = kaais.item_id and ia.address_full = kaais.address_full
                    inner join scp_b_202510.add_institution_lookup_kb inst
                        on kaais.inst_id_top = inst.inst_id
                    inner join scp_b_202510.add_institution_lookup_kb_suppl AS kb_inst
                    	on kaais.inst_id_top = kb_inst.inst_id
                    inner join unignhaupka.inst_with_federal_state_filled AS federal_states
                    	on case
                        	when kb_inst.inst_id = 621 then 'https://ror.org/03m2kj587'
                        else kb_inst.ror 
                      end = CONCAT('https://ror.org/', federal_states.ror_id)
                    where
                        item_type && array['Review', 'Article']
                        and federal_state = 'Niedersachsen' 
                        and ('uni' = any(current_sectors) or 'fh' = any(current_sectors) or 'khmh' = any(current_sectors))
                        and dfg_inst_id not in (
                        	220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                          	13033, --PFH Private Hochschule Göttingen
                          	233118106, -- Leibniz-Fachhochschule
                            198800578, -- Hochschule 21 Buxtehude
                            195374963 -- Hochschule Weserbergland
                      	)
                      	and pubyear between 2015 and 2025 
                     )
                     select inst_name, count(distinct(doi)) as n
                     from scopus
                     group by inst_name
                     order by n desc
                  """, 
                  con=engine)

In [78]:
scp_aff

,inst_name,n
0,Georg-August-Universität Göttingen,39605
1,Medizinische Hochschule Hannover (MHH),23322
2,Gottfried Wilhelm Leibniz Universität Hannover,17244
3,Technische Universität Braunschweig,12809
4,Carl von Ossietzky Universität Oldenburg,10678
5,Universität Osnabrück,4906
6,Stiftung Tierärztliche Hochschule Hannover,4837
7,Leuphana Universität Lüneburg,3725
8,Technische Universität Clausthal,3116
9,Stiftung Universität Hildesheim,1137


In [89]:
oal.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False).head()

,inst_name,count
1,Georg-August-Universität Göttingen,37501
11,Medizinische Hochschule Hannover (MHH),31371
2,Gottfried Wilhelm Leibniz Universität Hannover,23100
15,Technische Universität Braunschweig,17747
0,Carl von Ossietzky Universität Oldenburg,13187


In [80]:
kb.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False).head()

,inst_name,count
1,Georg-August-Universität Göttingen,39161
11,Medizinische Hochschule Hannover (MHH),28693
2,Gottfried Wilhelm Leibniz Universität Hannover,21164
15,Technische Universität Braunschweig,15494
0,Carl von Ossietzky Universität Oldenburg,11319


In [81]:
oal_kb.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False).head()

,inst_name,count
1,Georg-August-Universität Göttingen,46104
11,Medizinische Hochschule Hannover (MHH),31972
2,Gottfried Wilhelm Leibniz Universität Hannover,24275
15,Technische Universität Braunschweig,18447
0,Carl von Ossietzky Universität Oldenburg,13896


In [82]:
oal_kb_db.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False).head()

,inst_name,count
1,Georg-August-Universität Göttingen,39927
11,Medizinische Hochschule Hannover (MHH),28773
2,Gottfried Wilhelm Leibniz Universität Hannover,21737
15,Technische Universität Braunschweig,15721
0,Carl von Ossietzky Universität Oldenburg,11511


In [83]:
scp.groupby(['inst_name'])['n'].sum().reset_index(name='count').sort_values(['count'], ascending=False).head()

,inst_name,count
1,Georg-August-Universität Göttingen,39703
11,Medizinische Hochschule Hannover (MHH),23379
2,Gottfried Wilhelm Leibniz Universität Hannover,17291
15,Technische Universität Braunschweig,12844
0,Carl von Ossietzky Universität Oldenburg,10717


In [84]:
scp_aff.head()

,inst_name,n
0,Georg-August-Universität Göttingen,39605
1,Medizinische Hochschule Hannover (MHH),23322
2,Gottfried Wilhelm Leibniz Universität Hannover,17244
3,Technische Universität Braunschweig,12809
4,Carl von Ossietzky Universität Oldenburg,10678


In [108]:
oal_kb_aff_db.head()

,inst_name,n
0,Georg-August-Universität Göttingen,39926
1,Medizinische Hochschule Hannover (MHH),28773
2,Gottfried Wilhelm Leibniz Universität Hannover,21736
3,Technische Universität Braunschweig,15721
4,Carl von Ossietzky Universität Oldenburg,11511


### OpenAlex UNION Scopus (BDB)

In [102]:
scp_oal = pd.read_sql("""
                  WITH oal AS (
                        SELECT 
                          doi,
                          federal_states.inst_name,
                          oal.pubyear,
                          oal.class_name AS topic
                        FROM oal_b_20250711.items AS oal
                        JOIN oal_b_20250711.add_institution_kb_a AS kb_a
                          ON oal.item_id = kb_a.item_id
                        JOIN oal_b_20250711.add_institution_lookup_kb_suppl AS kb_inst
                          ON kb_a.inst_id_top = kb_inst.inst_id
                        JOIN unignhaupka.inst_with_federal_state_filled AS federal_states
                          ON CASE
                                WHEN kb_inst.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                                ELSE kb_inst.ror 
                              END = CONCAT('https://ror.org/', federal_states.ror_id)
                        WHERE item_type && ARRAY['review', 'article']
                          AND federal_state = 'Niedersachsen' 
                          AND ('uni' = ANY(current_sectors) OR 'fh' = ANY(current_sectors) OR 'khmh' = ANY(current_sectors))
                          AND dfg_inst_id NOT IN (
                              220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                              13033, --PFH Private Hochschule Göttingen
                              233118106, -- Leibniz-Fachhochschule
                              198800578, -- Hochschule 21 Buxtehude
                              195374963 -- Hochschule Weserbergland
                          )
                          AND pubyear BETWEEN 2015 AND 2025
                    ),
                     scopus AS (
                        SELECT 
                              scp.doi,
                              federal_states.inst_name,
                              scp.pubyear,
                              scp.class_name AS topic
                          FROM scp_b_202510.items AS scp
                          JOIN scp_b_202510.add_institution_kb_a AS kb_a
                              ON scp.item_id = kb_a.item_id
                          JOIN scp_b_202510.add_institution_lookup_kb_suppl AS kb_inst
                              ON kb_a.inst_id_top = kb_inst.inst_id
                          JOIN unignhaupka.inst_with_federal_state_filled AS federal_states
                              ON CASE
                                    WHEN kb_inst.inst_id = 621 THEN 'https://ror.org/03m2kj587'
                                    ELSE kb_inst.ror 
                                  END = CONCAT('https://ror.org/', federal_states.ror_id)
                          WHERE item_type && ARRAY['Review', 'Article']
                              AND federal_state = 'Niedersachsen' 
                              AND ('uni' = ANY(current_sectors) OR 'fh' = ANY(current_sectors) OR 'khmh' = ANY(current_sectors))
                              AND dfg_inst_id NOT IN (
                                  220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                                  13033, --PFH Private Hochschule Göttingen
                                  233118106, -- Leibniz-Fachhochschule
                                  198800578, -- Hochschule 21 Buxtehude
                                  195374963 -- Hochschule Weserbergland
                              )
                              AND pubyear BETWEEN 2015 AND 2025
                    )
                    SELECT
                        inst_name,
                        COUNT(DISTINCT(doi)) AS n
                    FROM (
                        SELECT doi, inst_name
                        FROM oal
                        UNION DISTINCT
                        SELECT doi, inst_name
                        FROM scopus
                     )
                     GROUP BY inst_name
                     ORDER BY n desc
                  """, 
                  con=engine)

In [103]:
scp_oal

,inst_name,n
0,Georg-August-Universität Göttingen,48283
1,Medizinische Hochschule Hannover (MHH),31687
2,Gottfried Wilhelm Leibniz Universität Hannover,24555
3,Technische Universität Braunschweig,17865
4,Carl von Ossietzky Universität Oldenburg,13777
5,Universität Osnabrück,6660
6,Stiftung Tierärztliche Hochschule Hannover,5788
7,Leuphana Universität Lüneburg,5078
8,Technische Universität Clausthal,4312
9,Stiftung Universität Hildesheim,1778


### OpenAlex UNION Scopus über AFF-String (BDB)

In [110]:
scp_oal_aff = pd.read_sql("""
                          with oal as (
                            select
                            	inst.name as inst_name,
                                doi
                            from oal_b_20250711.items x
                            inner join oal_b_20250711.items_authors via
                                on x.item_id = via.item_id
                            inner join oal_b_20250711.authors_affiliations vaa2
                                on x.item_id = vaa2.item_id and via.author_seq_nr = vaa2.author_seq_nr
                            inner join oal_b_20250711.items_affiliations ia
                                on x.item_id = ia.item_id and vaa2.address_full = ia.address_full
                            inner join oal_b_20250711.add_institution_kb_a kaais
                                on x.item_id = kaais.item_id and ia.address_full = kaais.address_full
                            inner join oal_b_20250711.add_institution_lookup_kb inst
                                on kaais.inst_id_top = inst.inst_id
                            inner join oal_b_20250711.add_institution_lookup_kb_suppl AS kb_inst
                            	on kaais.inst_id_top = kb_inst.inst_id
                            inner join unignhaupka.inst_with_federal_state_filled AS federal_states
                            	on case
                                	when kb_inst.inst_id = 621 then 'https://ror.org/03m2kj587'
                                else kb_inst.ror 
                              end = CONCAT('https://ror.org/', federal_states.ror_id)
                            where
                                item_type && array['article', 'review']
                                and federal_state = 'Niedersachsen' 
                                and ('uni' = any(current_sectors) or 'fh' = any(current_sectors) or 'khmh' = any(current_sectors))
                                and dfg_inst_id not in (
                                	220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                                  	13033, --PFH Private Hochschule Göttingen
                                  	233118106, -- Leibniz-Fachhochschule
                                    198800578, -- Hochschule 21 Buxtehude
                                    195374963 -- Hochschule Weserbergland
                              	)
                              	and pubyear between 2015 and 2025 
                          ),
                          scopus as (
                                select
                                	inst.name as inst_name,
                                    doi
                                from scp_b_202510.items x
                                inner join scp_b_202510.items_authors via
                                    on x.item_id = via.item_id
                                inner join scp_b_202510.authors_affiliations vaa2
                                    on x.item_id = vaa2.item_id and via.author_seq_nr = vaa2.author_seq_nr
                                inner join scp_b_202510.items_affiliations ia
                                    on x.item_id = ia.item_id and vaa2.aff_seq_nr = ia.aff_seq_nr
                                inner join scp_b_202510.add_institution_kb_a kaais
                                    on x.item_id = kaais.item_id and ia.address_full = kaais.address_full
                                inner join scp_b_202510.add_institution_lookup_kb inst
                                    on kaais.inst_id_top = inst.inst_id
                                inner join scp_b_202510.add_institution_lookup_kb_suppl AS kb_inst
                                	on kaais.inst_id_top = kb_inst.inst_id
                                inner join unignhaupka.inst_with_federal_state_filled AS federal_states
                                	on case
                                    	when kb_inst.inst_id = 621 then 'https://ror.org/03m2kj587'
                                    else kb_inst.ror 
                                  end = CONCAT('https://ror.org/', federal_states.ror_id)
                                where
                                    item_type && array['Review', 'Article']
                                    and federal_state = 'Niedersachsen' 
                                    and ('uni' = any(current_sectors) or 'fh' = any(current_sectors) or 'khmh' = any(current_sectors))
                                    and dfg_inst_id not in (
                                    	220269952, -- Fachhochschule für die Wirtschaft Hannover (FHDW)
                                      	13033, --PFH Private Hochschule Göttingen
                                      	233118106, -- Leibniz-Fachhochschule
                                        198800578, -- Hochschule 21 Buxtehude
                                        195374963 -- Hochschule Weserbergland
                                  	)
                                  	and pubyear between 2015 and 2025 
                         )
                        SELECT
                            inst_name,
                            COUNT(DISTINCT(doi)) AS n
                        FROM (
                            SELECT doi, inst_name
                            FROM oal
                            UNION DISTINCT
                            SELECT doi, inst_name
                            FROM scopus
                         )
                         GROUP BY inst_name
                         ORDER BY n desc
                  """, 
                  con=engine)

In [111]:
scp_oal_aff

,inst_name,n
0,Georg-August-Universität Göttingen,48221
1,Medizinische Hochschule Hannover (MHH),31651
2,Gottfried Wilhelm Leibniz Universität Hannover,24530
3,Technische Universität Braunschweig,17851
4,Carl von Ossietzky Universität Oldenburg,13746
5,Universität Osnabrück,6650
6,Stiftung Tierärztliche Hochschule Hannover,5759
7,Leuphana Universität Lüneburg,5068
8,Technische Universität Clausthal,4311
9,Stiftung Universität Hildesheim,1773


In [ ]:
get_domains = pd.read_sql("""
                    select doi, domains.display_name as domain_name
                    from oal_rep_20250711.works as oal
                    join oal_rep_20250711.works_topics as works_topics
                    	on oal.id = works_topics.work_id 
                    join oal_rep_20250711.topics as topics
                    	on works_topics.topic_id = topics.id
                    join oal_rep_20250711.domains as domains
                    	on topics.domain_id = domains.id
                    where publication_year between 2015 and 2025 
                  """, 
                  con=engine)